# DataMate: chạy model trên Colab

Trước khi chạy: trên máy local chạy `python scripts/pack.py`, tải `datamate.zip` lên thư mục gốc của Google Drive.
Chọn Runtime → Change runtime type → **A100** nếu có (nhanh và đủ bộ nhớ), không thì **L4**. Mỗi ô chạy một lần, theo thứ tự.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# datamate.zip ở bất kỳ thư mục nào của My Drive (sâu tối đa 3 cấp); datamate_code.zip (nếu có) đè code mới lên.
!Z=$(find /content/drive/MyDrive -maxdepth 3 -name datamate.zip | head -1) && rm -rf /content/dm && mkdir -p /content/dm && cd /content/dm && unzip -q "$Z" && echo "from $Z"
!Z=$(find /content/drive/MyDrive -maxdepth 3 -name datamate_code.zip | head -1); [ -n "$Z" ] && cd /content/dm && unzip -oq "$Z" && echo "code updated from $Z"; ls /content/dm

In [ ]:
# vLLM mới nhất: các model ra năm 2026 cần bản mới hơn base image của BTC. Ghi lại phiên bản để dùng khi build Docker.
!pip -q install vllm pandas==2.2.3 python-calamine==0.9.1
!pip show vllm | grep Version

In [ ]:
# Model ứng viên: đổi dòng này để so model. Tổng tham số phải <= 15B, weights công bố trước 26/10/2026.
MODEL = 'Qwen/Qwen3.5-9B'   # hoặc 'google/gemma-4-12B-it', 'Qwen/Qwen3-14B'
MAXLEN = 32768              # A100 80GB / G4 thừa sức. L4 24GB: hạ xuống 20000, hoặc số trong thông báo lỗi KV cache.
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir='/content/model')
!du -sh /content/model && nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess, time, urllib.request
log = open('/content/vllm.log', 'w')
srv = subprocess.Popen(['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', '/content/model',
                        '--served-model-name', 'model', '--port', '8000', '--max-model-len', str(MAXLEN), '--gpu-memory-utilization', '0.95', '--seed', '0', '--reasoning-parser', 'qwen3'],
                       stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3); break
    except Exception:
        assert srv.poll() is None, open('/content/vllm.log').read()[-3000:]
        time.sleep(5)
print(f'model san sang sau {time.time() - t0:.0f} giay')

## Đo trên tập train cố định

Không dùng manifest viết tay: đúng luồng sẽ chạy lúc thi. Ví dụ mẫu không lấy từ chính shop đang hỏi.

In [ ]:
%cd /content/dm
# Tập đo cố định (training_code/splits/README.md): dev300 để dò lỗi, holdout300 chỉ để báo điểm.
SPLIT = 'holdout300'
import time; t0 = time.time()
!python -X utf8 -m src.agent --data data/train/train --questions training_code/splits/{SPLIT}.jsonl --out /content/out_{SPLIT} --workers 8
print(f'{time.time() - t0:.0f} giay')
!python scripts/score.py /content/out_{SPLIT}/submission.jsonl --questions training_code/splits/{SPLIT}.jsonl --show 0

In [ ]:
# Xem các câu sai: câu hỏi, code model viết, kết quả, đáp án chuẩn.
import json, sys
sys.path.insert(0, '/content/dm')
from scripts.score import score_one
Q = {json.loads(l)['id']: json.loads(l) for l in open('/content/dm/data/train/train/questions.jsonl', encoding='utf-8')}
G = {json.loads(l)['id']: json.loads(l) for l in open('/content/dm/data/train/train/answers.jsonl', encoding='utf-8')}
for l in open('/content/out/logs/log.jsonl', encoding='utf-8'):
    r = json.loads(l)
    if score_one(r['answer'], G[r['id']]) != 1:
        print(Q[r['id']]['business_id'], '|', Q[r['id']]['question'])
        print('   code:', r['code'])
        print('   ra:', str(r['output'])[:150], '| dap an:', str(G[r['id']].get('answer'))[:150], G[r['id']]['type'])


## Chạy public test để nộp leaderboard

File kết quả được chép sang Drive: `submission_public.jsonl`.

In [ ]:
import time; t0 = time.time()
!python -X utf8 -m src.agent --data data/public_test/public_test --out /content/pub --workers 8
print(f'{time.time() - t0:.0f} giay cho public test')
!cp /content/pub/submission.jsonl /content/drive/MyDrive/submission_public.jsonl
!cp -r /content/pub/logs /content/drive/MyDrive/logs_public
import json, collections
print(collections.Counter(json.loads(l)['type'] for l in open('/content/pub/submission.jsonl', encoding='utf-8')))

In [ ]:
srv.terminate()  # tat model. Nho Runtime -> Disconnect and delete runtime de khoi ton unit.